# Regression Modeling Pitfalls

A fitted regression equation is useful only when the modeling process and assumptions make sense. This notebook summarizes the main pitfalls from the lecture and connects each one to a Python check.

By the end of this notebook, you should be able to:

- separate association from causation in observational regression;
- state the model-building cycle used in regression practice;
- identify seven common regression pitfalls before trusting a fitted model;
- run basic checks for estimability, plots, and assumption-sensitive interpretation.

In [ ]:
from lite_setup import ensure_packages
await ensure_packages()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from checks import check_columns

process = pd.read_csv(Path('data/process_diagnostics.csv'))
print(check_columns(process, ['YieldLoss', 'Temperature', 'LineSpeed', 'Pressure']))
process.head()

## The Regression Modeling Cycle

A practical regression cycle is:

1. Hypothesize a model form and collect data.
2. Make graphs before fitting.
3. Estimate parameters and specify an error model.
4. Check statistical significance.
5. Diagnose model form, assumptions, and unusual observations.
6. Change the model if needed and repeat.
7. Validate the model.
8. Use the model only in the region where it is defensible.

Fitting is not the end of the process. Regression is a loop: estimate, diagnose, revise, and validate.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, col in zip(axes, ['Temperature', 'LineSpeed', 'Pressure']):
    ax.scatter(process[col], process['YieldLoss'], alpha=0.8)
    ax.set_xlabel(col)
    ax.set_ylabel('YieldLoss')
plt.tight_layout()

## Seven Pitfalls

The lecture notes emphasize these pitfalls:

1. Assigning cause and effect from observational data.
2. Failing to randomize or collect representative data.
3. Ignoring deviations from error assumptions: mean zero, constant variance, normality when needed, and independence.
4. Trying to fit a model that is not estimable.
5. Missing multicollinearity.
6. Extrapolating outside the data region.
7. Ignoring outliers, high-leverage points, and influential points.

Interpretation: diagnostics do not prove a model is true. They help you find reasons the current model may be inadequate for the question.

In [ ]:
model_linear = smf.ols('YieldLoss ~ Temperature + LineSpeed + Pressure', data=process).fit()
print(model_linear.summary().tables[1])
print('R-squared:', round(model_linear.rsquared, 4))
print('Condition number:', round(model_linear.condition_number, 2))

## Estimability and Rank

A regression model is estimable when the design matrix has enough independent information to estimate the coefficients. If two columns are exact linear combinations of each other, or if there are too many parameters for the unique data patterns, the design matrix is rank deficient.

For a model matrix $X$, a basic check is:

$$\text{rank}(X) = \text{number of columns in } X.$$

If the rank is smaller than the number of columns, at least one coefficient is not separately estimable.

In [ ]:
X = model_linear.model.exog
rank = np.linalg.matrix_rank(X)
print('Design matrix shape:', X.shape)
print('Rank:', rank)
print('Full column rank?', rank == X.shape[1])

## Same Summary, Different Data: Anscombe's Lesson

The regression pitfalls lecture uses Anscombe's quartet to make an important diagnostic point: numerical summaries can look almost identical while the data patterns are completely different. Each dataset below has nearly the same fitted line, $R^2$, and hypothesis-test summaries, but only the plots reveal whether a straight-line regression is reasonable.

This is why a regression workflow should begin with graphs and return to graphs after fitting the model.

In [ ]:
anscombe = pd.DataFrame({
    'x1': [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
    'y1': [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68],
    'x2': [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
    'y2': [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74],
    'x3': [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
    'y3': [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73],
    'x4': [8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
    'y4': [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89],
})

summary_rows = []
fig, axes = plt.subplots(2, 2, figsize=(8, 6), sharex=True, sharey=True)
for idx, ax in zip(range(1, 5), axes.ravel()):
    x_col, y_col = f'x{idx}', f'y{idx}'
    fit = smf.ols(f'{y_col} ~ {x_col}', data=anscombe).fit()
    summary_rows.append({
        'dataset': idx,
        'intercept': fit.params['Intercept'],
        'slope': fit.params[x_col],
        'R_squared': fit.rsquared,
        'residual_std_error': np.sqrt(fit.mse_resid),
    })
    ax.scatter(anscombe[x_col], anscombe[y_col])
    x_grid = np.linspace(anscombe[x_col].min(), anscombe[x_col].max(), 100)
    ax.plot(x_grid, fit.predict(pd.DataFrame({x_col: x_grid})), color='black')
    ax.set_title(f'Dataset {idx}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
plt.tight_layout()

pd.DataFrame(summary_rows).round(3)

## Why Plots Come First

A large $R^2$ or significant F test can coexist with a poor model form, nonconstant variance, or influential observations. The rest of this module gives you concrete Python tools for those checks.

Transfer question: before reading the next notebook, list two plots you would want to see before using a regression model for prediction.